# G3 tournament shard 18 (forest)

Runs **223 cells** of the frozen Phase G3 manifest (`G3-MAIN-v1`), covering: `causal_drf`, `wdrft`.

This shard runs the two R forest baselines. The setup cell installs R and the pinned `drf` 1.3.1 and takes fifteen to twenty-five minutes; the Causal-DRF Rcpp unit is compiled once before the shard fans out, not per cell.

Measured compute for this shard on the reference machine is about **48 minutes** single-threaded. Colab cores are slower, so allow two to three times that, plus any install time above. This fits comfortably inside a nine hour session.

**Run every cell in order.** The last cell downloads a `.zip` containing this shard's results; collect all shards' zips into `results/main/` in the repository and run the merge.

The shard is idempotent: re-running the notebook recomputes the same cells from the same seeds and produces identical rows.

In [ ]:
# Thread pinning MUST happen before NumPy, SciPy or stochtree are
# imported. OpenMP sizes its pool at initialisation, so setting
# these afterwards is silently ineffective and costs roughly a
# factor of forty in throughput.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_v] = '1'
print('threads pinned to 1')

## 1. Unpack the source tree

In [ ]:
import base64, io, tarfile, pathlib

ARCHIVE = '''...'''

workdir = pathlib.Path('/content/wcf')
workdir.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(ARCHIVE))) as tar:
    tar.extractall(workdir)
os.chdir(workdir)
import sys
sys.path.insert(0, str(workdir / 'src'))
print('source ready at', workdir)
print(sorted(p.name for p in (workdir / 'src' / 'wasserstein_causal_forests').iterdir())[:12])

## 2. Dependencies

In [ ]:
# W-DRF-T uses the pinned CRAN drf 1.3.1; Causal-DRF compiles an Rcpp
# translation unit. Expect fifteen to twenty-five minutes here.
%%bash
set -e
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev > /dev/null 2>&1
Rscript -e 'options(Ncpus=2); install.packages(c("Rcpp","RcppEigen","jsonlite"), repos="https://cloud.r-project.org", quiet=TRUE)'
Rscript -e 'options(Ncpus=2); install.packages("https://cran.r-project.org/src/contrib/Archive/drf/drf_1.3.1.tar.gz", repos=NULL, type="source", quiet=TRUE)' || Rscript -e 'options(Ncpus=2); install.packages("drf", repos="https://cloud.r-project.org", quiet=TRUE)'
Rscript -e 'cat("drf", as.character(packageVersion("drf")), "ready\n")'


## 3. This shard's cells

In [ ]:
import json
SHARD_INDEX = 18
GROUP = 'forest'
CELLS = json.loads('''[{"grid": "scaling", "dgp": "D1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "fb7b9839209775f4", "test_seed": 900004}, {"grid": "scaling", "dgp": "D4", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "e48c90d19a17ad48", "test_seed": 900000}, {"grid": "scaling", "dgp": "D4", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "af9c78ec320cded4", "test_seed": 900006}, {"grid": "scaling", "dgp": "D6", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "046e0903bbc4b7ce", "test_seed": 900002}, {"grid": "scaling", "dgp": "D6", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "4104e03a1463e83c", "test_seed": 900008}, {"grid": "scaling", "dgp": "D1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "11254b3a1e2a0159", "test_seed": 900004}, {"grid": "scaling", "dgp": "D4", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "f8659952a43b80de", "test_seed": 900000}, {"grid": "scaling", "dgp": "D4", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "69ef7809cc6ccd37", "test_seed": 900006}, {"grid": "scaling", "dgp": "D6", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "19edd9a693ff5853", "test_seed": 900002}, {"grid": "scaling", "dgp": "D6", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "8dcee992e808d6a1", "test_seed": 900008}, {"grid": "main", "dgp": "D0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "255e23442ce26379", "test_seed": 900004}, {"grid": "main", "dgp": "D0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "ac26f10c3cfde582", "test_seed": 900010}, {"grid": "main", "dgp": "D0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "54f5497791ff77d1", "test_seed": 900016}, {"grid": "main", "dgp": "D1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "7dce125bf000b9b6", "test_seed": 900002}, {"grid": "main", "dgp": "D1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "a9d558878d86f720", "test_seed": 900008}, {"grid": "main", "dgp": "D1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 14, "cell_key": "1cd07dc02b23f832", "test_seed": 900014}, {"grid": "main", "dgp": "D2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "9323c2f31a2ad499", "test_seed": 900000}, {"grid": "main", "dgp": "D2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "9a6c1cb126d21a00", "test_seed": 900006}, {"grid": "main", "dgp": "D2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 12, "cell_key": "a98180a5f415808c", "test_seed": 900012}, {"grid": "main", "dgp": "D2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 18, "cell_key": "b2eff9e4e59c0088", "test_seed": 900018}, {"grid": "main", "dgp": "D3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "a14e3b6cb39f0223", "test_seed": 900004}, {"grid": "main", "dgp": "D3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "d1716d0291719cff", "test_seed": 900010}, {"grid": "main", "dgp": "D3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "148c3fdd0b352f55", "test_seed": 900016}, {"grid": "main", "dgp": "D4", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "06f70f2e96e6b64c", "test_seed": 900002}, {"grid": "main", "dgp": "D4", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "7a915238abff5bf8", "test_seed": 900008}, {"grid": "main", "dgp": "D4", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 14, "cell_key": "2604c7865f183bff", "test_seed": 900014}, {"grid": "main", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "08ac5612902634b1", "test_seed": 900000}, {"grid": "main", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "e433814ae1deadbb", "test_seed": 900006}, {"grid": "main", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 12, "cell_key": "408f9729b1875b76", "test_seed": 900012}, {"grid": "main", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 18, "cell_key": "a8239877b6551556", "test_seed": 900018}, {"grid": "main", "dgp": "D6", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "cc9a357791acc289", "test_seed": 900004}, {"grid": "main", "dgp": "D6", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "d4ece58a53518af9", "test_seed": 900010}, {"grid": "main", "dgp": "D6", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "09886050817b6bcf", "test_seed": 900016}, {"grid": "main", "dgp": "D7", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "b9ab28ced0f09ca2", "test_seed": 900002}, {"grid": "main", "dgp": "D7", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "5373fcda80d2266f", "test_seed": 900008}, {"grid": "main", "dgp": "D7", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 14, "cell_key": "1771c9fd928cc899", "test_seed": 900014}, {"grid": "main", "dgp": "D8", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "a354dc45e4029912", "test_seed": 900000}, {"grid": "main", "dgp": "D8", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "15db65f27e9f1f02", "test_seed": 900006}, {"grid": "main", "dgp": "D8", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 12, "cell_key": "329441cd6d8c4acf", "test_seed": 900012}, {"grid": "main", "dgp": "D8", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 18, "cell_key": "a919d982a1163874", "test_seed": 900018}, {"grid": "main", "dgp": "D9", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "626dd5adb81599c1", "test_seed": 900004}, {"grid": "main", "dgp": "D9", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "d8cc8a2e71182c15", "test_seed": 900010}, {"grid": "main", "dgp": "D9", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "25670a2062f6f1bc", "test_seed": 900016}, {"grid": "main", "dgp": "D0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "f87732478068caaa", "test_seed": 900002}, {"grid": "main", "dgp": "D0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "5f0995e98357cf9a", "test_seed": 900006}, {"grid": "main", "dgp": "D0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "acf506b6130abb51", "test_seed": 900012}, {"grid": "main", "dgp": "D0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "dfab0f2b3f49bb8f", "test_seed": 900018}, {"grid": "main", "dgp": "D1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "c28cdbe2a156aa6d", "test_seed": 900004}, {"grid": "main", "dgp": "D1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 10, "cell_key": "baa5855c243f161c", "test_seed": 900010}, {"grid": "main", "dgp": "D1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 16, "cell_key": "1abe20d655746b62", "test_seed": 900016}, {"grid": "main", "dgp": "D2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "cb57853994761e08", "test_seed": 900002}, {"grid": "main", "dgp": "D2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "a8bd11d24d17c5ac", "test_seed": 900008}, {"grid": "main", "dgp": "D2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "033487de08ed47ca", "test_seed": 900014}, {"grid": "main", "dgp": "D3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "e21c796d45ca0e3f", "test_seed": 900000}, {"grid": "main", "dgp": "D3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "6586fa49df8d203d", "test_seed": 900006}, {"grid": "main", "dgp": "D3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "8969bd2ffff9c260", "test_seed": 900012}, {"grid": "main", "dgp": "D3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "e7c33b10f1ccabbc", "test_seed": 900018}, {"grid": "main", "dgp": "D4", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "b2befde7eac50aad", "test_seed": 900004}, {"grid": "main", "dgp": "D4", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 10, "cell_key": "9c54a0f28e43f982", "test_seed": 900010}, {"grid": "main", "dgp": "D4", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 16, "cell_key": "bfb9f776e118f7e6", "test_seed": 900016}, {"grid": "main", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "a684635edaa1edc7", "test_seed": 900002}, {"grid": "main", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "a42b8ada26129080", "test_seed": 900008}, {"grid": "main", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "1a2facdb6e1f8c84", "test_seed": 900014}, {"grid": "main", "dgp": "D6", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "1bdf76c9cd9bf7c7", "test_seed": 900000}, {"grid": "main", "dgp": "D6", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "aadd5dd7d156858e", "test_seed": 900006}, {"grid": "main", "dgp": "D6", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "30015e574cbcd9dc", "test_seed": 900012}, {"grid": "main", "dgp": "D6", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "351bddf8931069a0", "test_seed": 900018}, {"grid": "main", "dgp": "D7", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "0e2d6ad9ec14fb04", "test_seed": 900004}, {"grid": "main", "dgp": "D7", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 10, "cell_key": "36ade56b7e0791d1", "test_seed": 900010}, {"grid": "main", "dgp": "D7", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 16, "cell_key": "82e8012a61ecfbcc", "test_seed": 900016}, {"grid": "main", "dgp": "D8", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "8a313e33a4369285", "test_seed": 900002}, {"grid": "main", "dgp": "D8", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "bba92bf2953fae26", "test_seed": 900008}, {"grid": "main", "dgp": "D8", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "683da4067d7608e4", "test_seed": 900014}, {"grid": "main", "dgp": "D9", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "4c87f3aef68f4bb4", "test_seed": 900000}, {"grid": "main", "dgp": "D9", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "2ff75183c433511e", "test_seed": 900006}, {"grid": "main", "dgp": "D9", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "62242d160c5ac2a9", "test_seed": 900012}, {"grid": "main", "dgp": "D9", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "13d2f577a7031930", "test_seed": 900018}, {"grid": "resolution", "dgp": "D1", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "a2c54b76d02e0eca", "test_seed": 900006}, {"grid": "resolution", "dgp": "D5", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "904ece86c15d7821", "test_seed": 900002}, {"grid": "resolution", "dgp": "D5", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "47334912a4a2f56b", "test_seed": 900008}, {"grid": "resolution", "dgp": "D6", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "98cfc586a3325f3d", "test_seed": 900004}, {"grid": "resolution", "dgp": "D7", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "06fca941b505147f", "test_seed": 900000}, {"grid": "resolution", "dgp": "D7", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "fabc3c07e8c2a526", "test_seed": 900006}, {"grid": "resolution", "dgp": "D1", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "3f5c9198e8c719d2", "test_seed": 900002}, {"grid": "resolution", "dgp": "D1", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "eb76f5453f6da2d7", "test_seed": 900008}, {"grid": "resolution", "dgp": "D5", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "b650bc93f932673e", "test_seed": 900004}, {"grid": "resolution", "dgp": "D6", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "35bf7a2d7f48e2fd", "test_seed": 900000}, {"grid": "resolution", "dgp": "D6", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "e3ef6d71c21a46d0", "test_seed": 900006}, {"grid": "resolution", "dgp": "D7", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "72ac4b8edbe9e6a0", "test_seed": 900002}, {"grid": "resolution", "dgp": "D7", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "243571cca92b6f4d", "test_seed": 900008}, {"grid": "main", "dgp": "D0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "5b0a40a0e0390979", "test_seed": 900004}, {"grid": "main", "dgp": "D0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "e5ea0db0b0a32815", "test_seed": 900010}, {"grid": "main", "dgp": "D0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "1ee064c4150687c7", "test_seed": 900016}, {"grid": "main", "dgp": "D1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "15a1a5be7d3d3c03", "test_seed": 900002}, {"grid": "main", "dgp": "D1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "817d97529e7c04e2", "test_seed": 900008}, {"grid": "main", "dgp": "D1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 14, "cell_key": "3b10ed32cccca6a3", "test_seed": 900014}, {"grid": "main", "dgp": "D2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "2d34e4dd4b1e6f4a", "test_seed": 900000}, {"grid": "main", "dgp": "D2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "fb056b0cbc6d22f5", "test_seed": 900006}, {"grid": "main", "dgp": "D2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 12, "cell_key": "8b3f79317fbe9724", "test_seed": 900012}, {"grid": "main", "dgp": "D2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 18, "cell_key": "748088cf2e28d865", "test_seed": 900018}, {"grid": "main", "dgp": "D3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "2dea941ab073cc1f", "test_seed": 900004}, {"grid": "main", "dgp": "D3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "a3e59f587c8164f5", "test_seed": 900010}, {"grid": "main", "dgp": "D3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "cd9b7bde2a5deb73", "test_seed": 900016}, {"grid": "main", "dgp": "D4", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "7419162b5ff14c19", "test_seed": 900002}, {"grid": "main", "dgp": "D4", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "94e99bb1d1e13a6b", "test_seed": 900008}, {"grid": "main", "dgp": "D4", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 14, "cell_key": "4325540c0df357bb", "test_seed": 900014}, {"grid": "main", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "10e6dce19ac50f9f", "test_seed": 900000}, {"grid": "main", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "3889192e9b62fa51", "test_seed": 900006}, {"grid": "main", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 12, "cell_key": "0c25b4c139fc5e78", "test_seed": 900012}, {"grid": "main", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 18, "cell_key": "ee0bd552e9fd3a02", "test_seed": 900018}, {"grid": "main", "dgp": "D6", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "060c65e88fd89dd2", "test_seed": 900004}, {"grid": "main", "dgp": "D6", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "164f041ef7257365", "test_seed": 900010}, {"grid": "main", "dgp": "D6", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "d618889ad79373e8", "test_seed": 900016}, {"grid": "main", "dgp": "D7", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "48ad1b6e1880d7cc", "test_seed": 900002}, {"grid": "main", "dgp": "D7", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "6591b8869bcc6567", "test_seed": 900008}, {"grid": "main", "dgp": "D7", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 14, "cell_key": "a8e5d879d4065ff4", "test_seed": 900014}, {"grid": "main", "dgp": "D8", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "0431ea69ea197789", "test_seed": 900000}, {"grid": "main", "dgp": "D8", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "000f42f2a1d48815", "test_seed": 900006}, {"grid": "main", "dgp": "D8", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 12, "cell_key": "ea1b5b87c21c2e03", "test_seed": 900012}, {"grid": "main", "dgp": "D8", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 18, "cell_key": "7e97edc35591fd59", "test_seed": 900018}, {"grid": "main", "dgp": "D9", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "25608a9c8950e4f3", "test_seed": 900004}, {"grid": "main", "dgp": "D9", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "ca574af293b9f5f6", "test_seed": 900010}, {"grid": "main", "dgp": "D9", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "e3592db8b950b322", "test_seed": 900016}, {"grid": "main", "dgp": "D0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "119f1671befb6966", "test_seed": 900002}, {"grid": "main", "dgp": "D0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "0488668007ddc34d", "test_seed": 900008}, {"grid": "main", "dgp": "D0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "1a76345c41232902", "test_seed": 900014}, {"grid": "main", "dgp": "D1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "e6256de67313e17b", "test_seed": 900000}, {"grid": "main", "dgp": "D1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "669f805795abedd2", "test_seed": 900006}, {"grid": "main", "dgp": "D1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "49d39441ca4e783a", "test_seed": 900012}, {"grid": "main", "dgp": "D1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "7808417487e6e332", "test_seed": 900018}, {"grid": "main", "dgp": "D2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "2faac43f1a790510", "test_seed": 900004}, {"grid": "main", "dgp": "D2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 10, "cell_key": "166f9955928d71e6", "test_seed": 900010}, {"grid": "main", "dgp": "D2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 16, "cell_key": "11902192d6414f56", "test_seed": 900016}, {"grid": "main", "dgp": "D3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "b2a19f4da5a94615", "test_seed": 900002}, {"grid": "main", "dgp": "D3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "1307b757006fba95", "test_seed": 900008}, {"grid": "main", "dgp": "D3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "e12b705bd350edb1", "test_seed": 900014}, {"grid": "main", "dgp": "D4", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "85019aea53b0253e", "test_seed": 900000}, {"grid": "main", "dgp": "D4", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "f6093336563dc996", "test_seed": 900006}, {"grid": "main", "dgp": "D4", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "9df13985ed3eebd8", "test_seed": 900012}, {"grid": "main", "dgp": "D4", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "986ce9fadeaf19ea", "test_seed": 900018}, {"grid": "main", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "3f64df79c2559473", "test_seed": 900004}, {"grid": "main", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 10, "cell_key": "dff1e1e5abcfee97", "test_seed": 900010}, {"grid": "main", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 16, "cell_key": "53c2302da4f54595", "test_seed": 900016}, {"grid": "main", "dgp": "D6", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "fe8b1923d7551610", "test_seed": 900002}, {"grid": "main", "dgp": "D6", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "ec26a4b9c2d867b6", "test_seed": 900008}, {"grid": "main", "dgp": "D6", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "7f6286f2f136995a", "test_seed": 900014}, {"grid": "main", "dgp": "D7", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "a9d23b2b91c436d2", "test_seed": 900000}, {"grid": "main", "dgp": "D7", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "7be15e22a02bb859", "test_seed": 900006}, {"grid": "main", "dgp": "D7", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "e294101b160e4bab", "test_seed": 900012}, {"grid": "main", "dgp": "D7", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "dfbf486515290698", "test_seed": 900018}, {"grid": "main", "dgp": "D8", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "1e811ddc5765a723", "test_seed": 900004}, {"grid": "main", "dgp": "D8", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 10, "cell_key": "e930cbf4a3f40295", "test_seed": 900010}, {"grid": "main", "dgp": "D8", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 16, "cell_key": "4dd7d19175435cab", "test_seed": 900016}, {"grid": "main", "dgp": "D9", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "0916651626c3c199", "test_seed": 900002}, {"grid": "main", "dgp": "D9", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "2e9733008819c76d", "test_seed": 900008}, {"grid": "main", "dgp": "D9", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "109038b92ac2b61c", "test_seed": 900014}, {"grid": "smallk", "dgp": "D0", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "bd9bea212b29fa21", "test_seed": 900000}, {"grid": "smallk", "dgp": "D0", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "e7fd72d5184711f9", "test_seed": 900004}, {"grid": "smallk", "dgp": "D0", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "dadd2a3a0295a6f4", "test_seed": 900010}, {"grid": "smallk", "dgp": "D0", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "017bfb9c08e04e0d", "test_seed": 900016}, {"grid": "smallk", "dgp": "D1", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "390a4d165f2d4e38", "test_seed": 900002}, {"grid": "smallk", "dgp": "D1", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "7a1a2912dfd27f80", "test_seed": 900008}, {"grid": "smallk", "dgp": "D1", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 14, "cell_key": "1b05224eacd9e5ba", "test_seed": 900014}, {"grid": "smallk", "dgp": "D2", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "118061badc335f54", "test_seed": 900000}, {"grid": "smallk", "dgp": "D2", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "ce6caed580ddcb21", "test_seed": 900006}, {"grid": "smallk", "dgp": "D2", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 12, "cell_key": "28a5fa06f4cb367d", "test_seed": 900012}, {"grid": "smallk", "dgp": "D2", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 18, "cell_key": "e5d89dce50c60225", "test_seed": 900018}, {"grid": "smallk", "dgp": "D3", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "985427169824b4fa", "test_seed": 900004}, {"grid": "smallk", "dgp": "D3", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "1d7734db5fea594b", "test_seed": 900010}, {"grid": "smallk", "dgp": "D3", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "bc0435cb1de827be", "test_seed": 900016}, {"grid": "smallk", "dgp": "D4", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "f20246d678de48ed", "test_seed": 900002}, {"grid": "smallk", "dgp": "D4", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "e1aedc91fa4462dc", "test_seed": 900008}, {"grid": "smallk", "dgp": "D4", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 14, "cell_key": "7a79db4d99e9cb60", "test_seed": 900014}, {"grid": "smallk", "dgp": "D5", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "981d41cc340f8e7a", "test_seed": 900000}, {"grid": "smallk", "dgp": "D5", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "9f0c3cf5f0ec92d9", "test_seed": 900006}, {"grid": "smallk", "dgp": "D5", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 12, "cell_key": "5245baf75d8f3ba3", "test_seed": 900012}, {"grid": "smallk", "dgp": "D5", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 18, "cell_key": "24a6b05395c7c1e6", "test_seed": 900018}, {"grid": "smallk", "dgp": "D6", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "a06c712e1e021841", "test_seed": 900004}, {"grid": "smallk", "dgp": "D6", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "98f30d33e3710d68", "test_seed": 900010}, {"grid": "smallk", "dgp": "D6", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "63226bc77089181f", "test_seed": 900016}, {"grid": "smallk", "dgp": "D7", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 2, "cell_key": "5001c01d63f0028b", "test_seed": 900002}, {"grid": "smallk", "dgp": "D7", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 8, "cell_key": "fe0353ba1bd4224b", "test_seed": 900008}, {"grid": "smallk", "dgp": "D7", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 14, "cell_key": "c2234969a8f93eb5", "test_seed": 900014}, {"grid": "smallk", "dgp": "D8", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 0, "cell_key": "64b5472340aef0c1", "test_seed": 900000}, {"grid": "smallk", "dgp": "D8", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 6, "cell_key": "6042b0c7b6697903", "test_seed": 900006}, {"grid": "smallk", "dgp": "D8", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 12, "cell_key": "dc998dd9ccdd1650", "test_seed": 900012}, {"grid": "smallk", "dgp": "D8", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 18, "cell_key": "10d9ee21d5d9f160", "test_seed": 900018}, {"grid": "smallk", "dgp": "D9", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 4, "cell_key": "f477c42cf41d1a43", "test_seed": 900004}, {"grid": "smallk", "dgp": "D9", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 10, "cell_key": "5097f799d6322ba4", "test_seed": 900010}, {"grid": "smallk", "dgp": "D9", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "wdrft", "seed": 16, "cell_key": "5bc2ab560d1e7fe5", "test_seed": 900016}, {"grid": "smallk", "dgp": "D0", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "777c94e5278b1b94", "test_seed": 900002}, {"grid": "smallk", "dgp": "D0", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "894627c34bf8b41c", "test_seed": 900008}, {"grid": "smallk", "dgp": "D0", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "da9f8696eb76ef0f", "test_seed": 900014}, {"grid": "smallk", "dgp": "D1", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "26af9af07c2b78ce", "test_seed": 900000}, {"grid": "smallk", "dgp": "D1", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "38be245b60b699d2", "test_seed": 900006}, {"grid": "smallk", "dgp": "D1", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "e9700bdf26e7baf6", "test_seed": 900012}, {"grid": "smallk", "dgp": "D1", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "77c7120c0de4a442", "test_seed": 900018}, {"grid": "smallk", "dgp": "D2", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "0d0ee455695c40c4", "test_seed": 900004}, {"grid": "smallk", "dgp": "D2", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 10, "cell_key": "af2f95429e5b2e93", "test_seed": 900010}, {"grid": "smallk", "dgp": "D2", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 16, "cell_key": "4bc4f20887f6cf22", "test_seed": 900016}, {"grid": "smallk", "dgp": "D3", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "9ebb03cf028c3422", "test_seed": 900002}, {"grid": "smallk", "dgp": "D3", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "4d85c8f777e5d110", "test_seed": 900008}, {"grid": "smallk", "dgp": "D3", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "4077f4d5cad42ec5", "test_seed": 900014}, {"grid": "smallk", "dgp": "D4", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "a8c6d5feccd5cf4c", "test_seed": 900000}, {"grid": "smallk", "dgp": "D4", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "e6fb62fa132a8733", "test_seed": 900006}, {"grid": "smallk", "dgp": "D4", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "15d5c5511dae28e5", "test_seed": 900012}, {"grid": "smallk", "dgp": "D4", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "27a67d572b521dc0", "test_seed": 900018}, {"grid": "smallk", "dgp": "D5", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "6735bb89125c6b5d", "test_seed": 900004}, {"grid": "smallk", "dgp": "D5", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 10, "cell_key": "db9a7eaf8dd06a6b", "test_seed": 900010}, {"grid": "smallk", "dgp": "D5", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 16, "cell_key": "7bf24c092e1d8e5b", "test_seed": 900016}, {"grid": "smallk", "dgp": "D6", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "bf2ae6b94d1438df", "test_seed": 900002}, {"grid": "smallk", "dgp": "D6", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "df54ffb96cdd66ff", "test_seed": 900008}, {"grid": "smallk", "dgp": "D6", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "3fddef2ca6f80932", "test_seed": 900014}, {"grid": "smallk", "dgp": "D7", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "6cf81a93a93c0e55", "test_seed": 900000}, {"grid": "smallk", "dgp": "D7", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 6, "cell_key": "c25a5ca4fdb00675", "test_seed": 900006}, {"grid": "smallk", "dgp": "D7", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 12, "cell_key": "7d955073b5556323", "test_seed": 900012}, {"grid": "smallk", "dgp": "D7", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "0df5487393002b74", "test_seed": 900018}, {"grid": "smallk", "dgp": "D8", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "ba7ed0f890878c5d", "test_seed": 900004}, {"grid": "smallk", "dgp": "D8", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 10, "cell_key": "20d44a44bb29fb62", "test_seed": 900010}, {"grid": "smallk", "dgp": "D8", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 16, "cell_key": "e43d8acf91ab685a", "test_seed": 900016}, {"grid": "smallk", "dgp": "D9", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "c25d09f50f06a008", "test_seed": 900002}, {"grid": "smallk", "dgp": "D9", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "73df38aaaf29562c", "test_seed": 900008}, {"grid": "smallk", "dgp": "D9", "n_train": 500, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 14, "cell_key": "a0c1df746b67aaf0", "test_seed": 900014}]''')
print(f'{len(CELLS)} cells in this shard')
import collections
for key, count in sorted(collections.Counter(
        (c['grid'], c['method']) for c in CELLS).items()):
    print(f'  {key[0]:12s} {key[1]:18s} {count}')

## 4. Run

In [ ]:
import time
from pathlib import Path
from wasserstein_causal_forests.g3.manifest import Cell
from wasserstein_causal_forests.g3.runner import run_shard

from wasserstein_causal_forests.g3 import r_bridge
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)
# Compile the Causal-DRF Rcpp unit once, before any cell runs.
try:
    r_bridge.warm_rcpp_cache(cache)
    print('Rcpp cache warm')
except Exception as error:
    print('cache warm failed, cells will report it:', error)

cells = [Cell(**{k: v for k, v in item.items()
                 if k not in ('cell_key', 'test_seed')})
         for item in CELLS]

out = Path('/content/wcf/results/main')
out.mkdir(parents=True, exist_ok=True)
log = Path(f'/content/wcf/results/manifests/execution_log_{SHARD_INDEX:03d}.jsonl')
log.parent.mkdir(parents=True, exist_ok=True)

started = time.time()
summary = run_shard(
    cells,
    out / f'shard_{SHARD_INDEX:03d}.parquet',
    cache_directory=cache,
    log_path=log,
)
print(json.dumps(summary, indent=2))
print(f'elapsed {(time.time() - started) / 60:.1f} min')

## 5. Check

Every cell must appear exactly once, as a success or as a failure. Failures are kept: the merge reconciles them against the manifest and reports them, and a seed is never silently replaced.

In [ ]:
import collections
records = [json.loads(line) for line in
           open(log, encoding='utf-8') if line.strip()]
status = collections.Counter(r['status'] for r in records)
print('cells logged:', len(records), '| expected:', len(CELLS))
print('status:', dict(status))
assert len(records) == len(CELLS), 'shard did not finish every cell'
for record in records:
    if record['status'] != 'ok':
        print('  FAILED', record['dgp'], record['method'],
              record['seed'])
slowest = sorted(records, key=lambda r: -r['wall_seconds'])[:5]
print('slowest cells:', [(r['method'], round(r['wall_seconds'], 1))
                         for r in slowest])

## 6. Download the results

In [ ]:
import shutil
bundle = f'/content/g3_shard_18_forest'
staging = Path('/content/bundle')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copy(out / f'shard_{SHARD_INDEX:03d}.parquet', staging)
shutil.copy(log, staging)
output_file = shutil.make_archive(bundle, 'zip', staging)
print('bundle:', output_file,
      f'({os.path.getsize(output_file) / 1e6:.2f} MB)')

try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)